# Feature Store Patterns — AI Lab Instructor Notebook

*System Design · Expert*



6 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
import pandas as pd

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Feature Store Patterns — Student Lab

Offline lab: point-in-time features (offline) vs online materialization; detect leakage + skew.

## 0 — Synthetic event log + labels

In [ ]:
import numpy as np
import pandas as pd

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Feature Store Patterns — Instructor Lab

Offline lab: point-in-time features (offline) vs online materialization; detect leakage + skew.

## 0 — Synthetic event log + labels

## Task 2: 0 — Synthetic event log + labels

## 0 — Synthetic event log + labels

In [ ]:
$f

In [ ]:
# Checks
check('events_nonempty', len(events) > 0)
check('labels_have_cols', set(['user_id','churned']).issubset(labels.columns))
check('labels_unique_users', labels['user_id'].is_unique)

## Task 3: 1 — Offline store: point-in-time features (no leakage)

## 1 — Offline store: point-in-time features (no leakage)

In [ ]:
def point_in_time_features(events: pd.DataFrame, asof: np.datetime64):
    hist = events[events["ts"] <= asof].copy()
    w7 = asof - np.timedelta64(7, "D")
    w30 = asof - np.timedelta64(30, "D")
    login7 = hist[(hist["event"]=="login") & (hist["ts"] > w7)].groupby("user_id").size()
    spend30 = hist[(hist["event"]=="purchase") & (hist["ts"] > w30)].groupby("user_id")["amount"].sum()
    out = pd.DataFrame({"user_id": events["user_id"].unique()})
    out["login_7d"] = out["user_id"].map(login7).fillna(0).astype(int)
    out["spend_30d"] = out["user_id"].map(spend30).fillna(0.0).astype(float)
    return out

X_off = point_in_time_features(events, cutoff)
df = labels.merge(X_off, on="user_id", how="left")
check('no_nan', df[['login_7d','spend_30d']].notnull().all().all())
print(df.head())


In [ ]:
# Checks
check('no_nan', df[['login_7d','spend_30d']].notnull().all().all())

## Task 4: 2 — Bad offline features (leaky)

## 2 — Bad offline features (leaky)

In [ ]:
def leaky_features(events: pd.DataFrame, asof: np.datetime64):
    # BUG: does not filter future events
    w7 = asof - np.timedelta64(7, "D")
    w30 = asof - np.timedelta64(30, "D")
    login7 = events[(events["event"]=="login") & (events["ts"] > w7)].groupby("user_id").size()
    spend30 = events[(events["event"]=="purchase") & (events["ts"] > w30)].groupby("user_id")["amount"].sum()
    out = pd.DataFrame({"user_id": events["user_id"].unique()})
    out["login_7d"] = out["user_id"].map(login7).fillna(0).astype(int)
    out["spend_30d"] = out["user_id"].map(spend30).fillna(0.0).astype(float)
    return out

X_leak = leaky_features(events, cutoff)
df_leak = labels.merge(X_leak, on="user_id", how="left")
print('avg_login7d_off_vs_leak', float(df['login_7d'].mean()), float(df_leak['login_7d'].mean()))


## Task 5: 3 — Quick evaluation: leakage inflates metrics

## 3 — Quick evaluation: leakage inflates metrics

In [ ]:
def simple_score(df: pd.DataFrame):
    x = 0.8*df["login_7d"].values + 0.02*df["spend_30d"].values
    return -x

def auc_like(scores: np.ndarray, y: np.ndarray, seed: int = 0):
    rng = np.random.default_rng(seed)
    pos = scores[y==1]
    neg = scores[y==0]
    if pos.size == 0 or neg.size == 0:
        return 0.0
    pos_s = rng.choice(pos, size=min(400, pos.size), replace=False)
    neg_s = rng.choice(neg, size=min(400, neg.size), replace=False)
    cnt = 0
    tot = 0
    for a in pos_s:
        cnt += int(np.sum(a > neg_s))
        tot += neg_s.size
    return float(cnt / max(1, tot))

y = df["churned"].astype(int).values
s_off = simple_score(df)
s_leak = simple_score(df_leak)
print('auc_like_off', auc_like(s_off, y))
print('auc_like_leak', auc_like(s_leak, y))


## Task 6: 4 — Online store simulation + skew check

## 4 — Online store simulation + skew check

In [ ]:
def materialize_daily(events: pd.DataFrame, day: np.datetime64):
    # Simulate an online store snapshot materialized near the end of a given day.
    return point_in_time_features(events, day + np.timedelta64(23, "h"))

asof_online = cutoff - np.timedelta64(1, "D")
X_online = materialize_daily(events, asof_online)
cmp = X_off.merge(X_online, on="user_id", suffixes=("_off", "_on"))
cmp["login_diff"] = cmp["login_7d_off"] - cmp["login_7d_on"]
print('max_abs_login_diff', int(cmp['login_diff'].abs().max()))
print('explanation', 'offline features were computed at the cutoff, but the online snapshot was materialized one day earlier, so the two pipelines refer to different as-of times and can disagree even without a code bug.')

**Why this works.** offline and online features can differ even without explicit bugs if they are computed at different as-of times.